# 09 Diagnostic Decomposition v1

This stage diagnoses why the current portfolio pipeline underperforms broad benchmarks despite the RL overlay slightly improving on the static allocator.

The core questions are:
- Is RL's value genuinely dynamic state-dependent control, or mostly a better fixed parameter pair?
- Where is performance being lost in the stock-level pipeline?
- Is the bottleneck more likely signal quality, optimizer configuration, or transaction-cost drag?

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
PYTHON = sys.executable
TRAIN_START = "2006-01"
TRAIN_END = "2016-12"
VAL_START = "2017-01"
VAL_END = "2019-12"
TEST_START = "2020-01"
TEST_END = "2024-11"
COST_BPS = 10.0
FIXED_PARAM_SOURCE = "validation_mean"

PRED_FILE = PROJECT_ROOT / "data/prediction/fm_oos_predictions.parquet"
PANEL_FILE = PROJECT_ROOT / "data/panel/monthly_stock_panel_basic_full.parquet"
RISK_DIR = PROJECT_ROOT / "data/risk/risk_cov_npz"
RISK_META_FILE = PROJECT_ROOT / "data/risk/risk_monthly_metadata.csv"
STATIC_BACKTEST = PROJECT_ROOT / "data/allocator/static_allocator_backtest.csv"
STATIC_WEIGHTS = PROJECT_ROOT / "data/allocator/static_allocator_weights.parquet"
RL_BACKTEST = PROJECT_ROOT / "data/rl_overlay_sac/test_backtest.csv"
RL_WEIGHTS = PROJECT_ROOT / "data/rl_overlay_sac/test_weights.parquet"
TRAIN_ACTIONS = PROJECT_ROOT / "data/rl_overlay_sac/train_action_history.csv"
VALIDATION_ACTIONS = PROJECT_ROOT / "data/rl_overlay_sac/validation_action_history.csv"
BENCHMARK_SUMMARY = PROJECT_ROOT / "data/benchmark_eval/benchmark_summary.csv"
BENCHMARK_RETURNS = PROJECT_ROOT / "data/benchmark_eval/benchmark_monthly_returns.csv"

STATIC_FIXED_FAIR_DIR = PROJECT_ROOT / "data/diagnostics/static_fixed_param_fair"
PORTFOLIO_DIAG_DIR = PROJECT_ROOT / "data/diagnostics/portfolio"
SIGNAL_DIAG_DIR = PROJECT_ROOT / "data/diagnostics/signal"

for path in [STATIC_FIXED_FAIR_DIR, PORTFOLIO_DIAG_DIR, SIGNAL_DIAG_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
required_artifacts = {
    "predictions": PRED_FILE,
    "panel": PANEL_FILE,
    "static_backtest": STATIC_BACKTEST,
    "static_weights": STATIC_WEIGHTS,
    "rl_backtest": RL_BACKTEST,
    "rl_weights": RL_WEIGHTS,
    "benchmark_summary": BENCHMARK_SUMMARY,
    "benchmark_monthly_returns": BENCHMARK_RETURNS,
}

if FIXED_PARAM_SOURCE == "train_mean":
    required_artifacts["train_action_history"] = TRAIN_ACTIONS
elif FIXED_PARAM_SOURCE == "validation_mean":
    required_artifacts["validation_action_history"] = VALIDATION_ACTIONS


status = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "required": True,
            "exists": path.exists(),
        }
        for name, path in required_artifacts.items()
    ]
)
display(status)

missing = status.loc[status["required"] & ~status["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required upstream artifacts: {missing}")


## Fair Fixed-Parameter Benchmark

This benchmark reruns the same long-only mean-variance allocator, but fixes lambda and tau using information available before the test window. The old diagnostic test-informed benchmark used the mean SAC actions from the test action history, which is useful for decomposition but not fair as a main benchmark. This revised fair fixed-parameter benchmark uses train- or validation-only parameter selection, with `validation_mean` as the default, then evaluates the fixed pair only on the test period.


In [ ]:
subprocess.run(
    [
        PYTHON,
        "scripts/run_static_fixed_parameter_benchmark.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--pred-file",
        str(PRED_FILE),
        "--risk-dir",
        str(RISK_DIR),
        "--risk-meta-file",
        str(RISK_META_FILE),
        "--returns-file",
        str(PANEL_FILE),
        "--train-action-history-file",
        str(TRAIN_ACTIONS),
        "--validation-action-history-file",
        str(VALIDATION_ACTIONS),
        "--fixed-param-source",
        FIXED_PARAM_SOURCE,
        "--outdir",
        str(STATIC_FIXED_FAIR_DIR),
        "--cost-bps",
        str(COST_BPS),
        "--test-start",
        TEST_START,
        "--test-end",
        TEST_END,
    ],
    check=True,
)


In [ ]:
benchmark_summary = pd.read_csv(BENCHMARK_SUMMARY)
static_fixed_fair_summary = pd.read_csv(STATIC_FIXED_FAIR_DIR / "static_fixed_fair_summary.csv")

comparison = pd.concat(
    [benchmark_summary, static_fixed_fair_summary],
    ignore_index=True,
    sort=False,
)
strategy_order = [
    "equal_weight",
    "spy_buy_hold",
    "static_allocator",
    "rl_overlay_sac",
    "static_fixed_param_fair",
]
comparison = comparison.set_index("strategy").reindex(strategy_order).reset_index()
display(
    comparison[
        [
            "strategy",
            "annualized_return",
            "annualized_volatility",
            "sharpe_ratio",
            "max_drawdown",
            "cumulative_return",
            "mean_monthly_turnover",
            "n_months",
        ]
    ]
)

display(
    static_fixed_fair_summary[
        [
            "fixed_param_source",
            "parameter_source_file",
            "fixed_lambda",
            "fixed_tau",
        ]
    ]
)


## Portfolio Diagnostics

These diagnostics decompose strategy behavior into gross versus net return, cost drag, turnover, holdings count, concentration, and drawdowns. The goal is to see whether underperformance is coming from costs, sparse/concentrated portfolios, or loss episodes.

In [ ]:
portfolio_jobs = [
    ("static_allocator", STATIC_BACKTEST, STATIC_WEIGHTS),
    ("rl_overlay_sac", RL_BACKTEST, RL_WEIGHTS),
    (
        "static_fixed_param_fair",
        STATIC_FIXED_FAIR_DIR / "static_fixed_fair_backtest.csv",
        STATIC_FIXED_FAIR_DIR / "static_fixed_fair_weights.parquet",
    ),
]

for strategy_name, backtest_file, weights_file in portfolio_jobs:
    subprocess.run(
        [
            PYTHON,
            "scripts/run_portfolio_diagnostics_v1.py",
            "--strategy-name",
            strategy_name,
            "--backtest-file",
            str(backtest_file),
            "--weights-file",
            str(weights_file),
            "--outdir",
            str(PORTFOLIO_DIAG_DIR),
            "--start-month",
            TEST_START,
            "--end-month",
            TEST_END,
        ],
        check=True,
    )


In [ ]:
portfolio_summaries = []
for strategy_name, _, _ in portfolio_jobs:
    portfolio_summaries.append(
        pd.read_csv(PORTFOLIO_DIAG_DIR / f"{strategy_name}_portfolio_diag_summary.csv")
    )

portfolio_summary = pd.concat(portfolio_summaries, ignore_index=True)
display(
    portfolio_summary[
        [
            "strategy",
            "mean_turnover",
            "mean_cost_drag",
            "mean_n_holdings",
            "mean_effective_n",
            "mean_max_weight",
            "mean_top10_weight_share",
            "max_drawdown",
        ]
    ]
)

## Signal Diagnostics by Train / Validation / Test

The prediction layer is evaluated directly using monthly IC, rank IC, top-minus-bottom decile spread, and mu_hat dispersion. This section reports the same diagnostics separately for train, validation, and test so we can see whether signal degradation is isolated to the test period or already visible before test.


In [ ]:
subprocess.run(
    [
        PYTHON,
        "scripts/run_signal_diagnostics_v1.py",
        "--pred-file",
        str(PRED_FILE),
        "--outdir",
        str(SIGNAL_DIAG_DIR),
        "--train-start",
        TRAIN_START,
        "--train-end",
        TRAIN_END,
        "--val-start",
        VAL_START,
        "--val-end",
        VAL_END,
        "--test-start",
        TEST_START,
        "--test-end",
        TEST_END,
    ],
    check=True,
)


In [ ]:
signal_split_summary = pd.read_csv(SIGNAL_DIAG_DIR / "signal_split_summary.csv")

# The summary file is already ordered with test first, then validation, then train.
display(
    signal_split_summary[
        [
            "split",
            "n_months",
            "mean_ic",
            "mean_ic_minus_test",
            "mean_rank_ic",
            "mean_rank_ic_minus_test",
            "mean_top_bottom_spread",
            "mean_top_bottom_spread_minus_test",
            "spread_sharpe",
            "average_mu_std",
            "average_mu_std_minus_test",
            "average_p90_minus_p10",
            "average_p90_minus_p10_minus_test",
            "average_n_obs",
        ]
    ]
)


## Interpretation Notes

- Compare `static_fixed_param_fair` against `rl_overlay_sac`. Similar performance suggests the SAC policy's test-period edge may be reproducible with a fixed pair selected ex ante; a material SAC edge suggests dynamic control is adding value.
- If supplied manually to the script, optional test-action means are diagnostics only. They are not used for selecting the fair benchmark parameters, and this notebook does not pass test action history in the fair benchmark call.
- If the test IC, rank IC, or top-bottom spread is weak while train and validation are stronger, the signal may be failing specifically out of sample in the test regime.
- If validation already deteriorates relative to train, the signal weakness is visible before the final test period and should be treated as model instability rather than a test-only surprise.
- If train looks materially different from test in both IC and mu_hat dispersion, the bottleneck may be nonstationarity, signal compression, or a training-window-specific relationship that does not persist.
- If gross returns are acceptable but net returns decay, cost drag and turnover are likely first-order portfolio bottlenecks.
- If the signal is positive but portfolios are overly concentrated, too sparse, or too conservative, the optimizer configuration is a natural next target.
- The next ablation notebook should test no-cost allocation, lower turnover penalties, max-weight constraints, signal-strength filters, and fixed grids selected on train/validation only.


In [ ]:
plots = [
    PROJECT_ROOT / "data/benchmark_eval/cumulative_nav_comparison.png",
    STATIC_FIXED_FAIR_DIR / "static_fixed_fair_cumret.png",
    PORTFOLIO_DIAG_DIR / "rl_overlay_sac_cost_drag.png",
    PORTFOLIO_DIAG_DIR / "static_allocator_cost_drag.png",
    PORTFOLIO_DIAG_DIR / "static_fixed_param_fair_cost_drag.png",
    SIGNAL_DIAG_DIR / "signal_ic_timeseries_by_split.png",
    SIGNAL_DIAG_DIR / "signal_top_bottom_spread_by_split.png",
]

for plot_path in plots:
    if plot_path.exists():
        print(plot_path.relative_to(PROJECT_ROOT))
        display(Image(filename=str(plot_path)))
